<a href="https://colab.research.google.com/github/Arya-code2005/Machine_Learning/blob/voting-ensemble/voting_regressor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import numpy as np
import pandas as pd # pandas is required for the manual data loading workaround

def load_boston(return_X_y=False):
    # This custom load_boston function replaces the deprecated scikit-learn version.
    # It uses the workaround suggested in the ImportError traceback to fetch the data.

    data_url = "http://lib.stat.cmu.edu/datasets/boston"
    raw_df = pd.read_csv(data_url, sep=r"\s+", skiprows=22, header=None)
    data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
    target = raw_df.values[1::2, 2]

    if return_X_y:
        return data, target
    else:
        # Mimic sklearn.utils.Bunch object for backward compatibility
        class Bunch(dict):
            def __init__(self, **kwargs):
                super().__init__(kwargs)
            def __getattr__(self, key):
                try:
                    return self[key]
                except KeyError:
                    raise AttributeError(key)
            def __setattr__(self, key, value):
                self[key] = value

        return Bunch(
            data=data,
            target=target,
            DESCR="Boston Housing Dataset (Manually loaded due to scikit-learn deprecation - use with caution for educational purposes on ethical issues)",
            feature_names=[f'feature_{i}' for i in range(data.shape[1])] # Dummy feature names
        )


In [6]:
X,y = load_boston(return_X_y=True)

In [7]:
X.shape

(506, 13)

In [8]:
y.shape

(506,)

In [9]:

X

array([[6.3200e-03, 1.8000e+01, 2.3100e+00, ..., 1.5300e+01, 3.9690e+02,
        4.9800e+00],
       [2.7310e-02, 0.0000e+00, 7.0700e+00, ..., 1.7800e+01, 3.9690e+02,
        9.1400e+00],
       [2.7290e-02, 0.0000e+00, 7.0700e+00, ..., 1.7800e+01, 3.9283e+02,
        4.0300e+00],
       ...,
       [6.0760e-02, 0.0000e+00, 1.1930e+01, ..., 2.1000e+01, 3.9690e+02,
        5.6400e+00],
       [1.0959e-01, 0.0000e+00, 1.1930e+01, ..., 2.1000e+01, 3.9345e+02,
        6.4800e+00],
       [4.7410e-02, 0.0000e+00, 1.1930e+01, ..., 2.1000e+01, 3.9690e+02,
        7.8800e+00]])

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score


In [11]:

lr = LinearRegression()
dt = DecisionTreeRegressor()
svr = SVR()


In [12]:
estimators = [('lr',lr),('dt',dt),('svr',svr)]

In [13]:

for estimator in estimators:
  scores = cross_val_score(estimator[1],X,y,scoring='r2',cv=10)
  print(estimator[0],np.round(np.mean(scores),2))

lr 0.2
dt -0.03
svr -0.41


In [14]:
from sklearn.ensemble import VotingRegressor

In [15]:

vr = VotingRegressor(estimators)
scores = cross_val_score(vr,X,y,scoring='r2',cv=10)
print("Voting Regressor",np.round(np.mean(scores),2))

Voting Regressor 0.41


In [16]:
for i in range(1,4):
  for j in range(1,4):
    for k in range(1,4):
      vr = VotingRegressor(estimators,weights=[i,j,k])
      scores = cross_val_score(vr,X,y,scoring='r2',cv=10)
      print("For i={},j={},k={}".format(i,j,k),np.round(np.mean(scores),2))


For i=1,j=1,k=1 0.45
For i=1,j=1,k=2 0.33
For i=1,j=1,k=3 0.26
For i=1,j=2,k=1 0.4
For i=1,j=2,k=2 0.32
For i=1,j=2,k=3 0.35
For i=1,j=3,k=1 0.28
For i=1,j=3,k=2 0.37
For i=1,j=3,k=3 0.35
For i=2,j=1,k=1 0.43
For i=2,j=1,k=2 0.42
For i=2,j=1,k=3 0.34
For i=2,j=2,k=1 0.44
For i=2,j=2,k=2 0.44
For i=2,j=2,k=3 0.42
For i=2,j=3,k=1 0.43
For i=2,j=3,k=2 0.44
For i=2,j=3,k=3 0.4
For i=3,j=1,k=1 0.43
For i=3,j=1,k=2 0.43
For i=3,j=1,k=3 0.4
For i=3,j=2,k=1 0.46
For i=3,j=2,k=2 0.45
For i=3,j=2,k=3 0.44
For i=3,j=3,k=1 0.35
For i=3,j=3,k=2 0.44
For i=3,j=3,k=3 0.45


In [17]:
# using the same algorithm

dt1 = DecisionTreeRegressor(max_depth=1)
dt2 = DecisionTreeRegressor(max_depth=3)
dt3 = DecisionTreeRegressor(max_depth=5)
dt4 = DecisionTreeRegressor(max_depth=7)
dt5 = DecisionTreeRegressor(max_depth=None)

In [18]:
estimators = [('dt1',dt1),('dt2',dt2),('dt3',dt3),('dt4',dt4),('dt5',dt5)]

In [19]:
for estimator in estimators:
  scores = cross_val_score(estimator[1],X,y,scoring='r2',cv=10)
  print(estimator[0],np.round(np.mean(scores),2))

dt1 -0.85
dt2 -0.11
dt3 -0.04
dt4 0.05
dt5 -0.3


In [20]:
vr = VotingRegressor(estimators)
scores = cross_val_score(vr,X,y,scoring='r2',cv=10)
print("Voting Regressor",np.round(np.mean(scores),2))


Voting Regressor 0.2
